# Multi-Model Image Generation

Compare image generation across 3 AI models using the same prompt.

**Before running:** get a free Hugging Face token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and paste it below.

In [ ]:
!pip install -q Pillow requests matplotlib

In [ ]:
import requests, io, time
from PIL import Image
import matplotlib.pyplot as plt

HF_TOKEN = "paste-your-token-here"
HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"}

PROMPT = "a flat illustration of a smiling merchant reviewing analytics on a tablet in a retail store, minimal style, no text, no watermark"

models = {
    "Stable Diffusion XL": "stabilityai/stable-diffusion-xl-base-1.0",
    "Flux Schnell": "black-forest-labs/FLUX.1-schnell",
    "SD 3 Medium": "stabilityai/stable-diffusion-3-medium-diffusers",
}

def generate(model_id, prompt):
    url = f"https://router.huggingface.co/hf-inference/models/{model_id}"
    for _ in range(3):
        r = requests.post(url, headers=HEADERS, json={"inputs": prompt}, timeout=120)
        if r.status_code == 200:
            return Image.open(io.BytesIO(r.content))
        if r.status_code == 401:
            print("  ✗ Invalid token. Check your HF_TOKEN above.")
            return "bad_token"
        if r.status_code == 402:
            print("  ✗ Free API limit reached. Try again later.")
            return None
        if r.status_code == 503:
            time.sleep(r.json().get("estimated_time", 30))
            continue
        print(f"  ✗ Error {r.status_code}")
        return None
    return None

stop = False
images = {}
for name, mid in models.items():
    if stop:
        break
    print(f"Generating with {name}...")
    result = generate(mid, PROMPT)
    if result == "bad_token":
        print("\n⚠️  Your HF_TOKEN is invalid. Go to https://huggingface.co/settings/tokens and paste a valid token above.")
        stop = True
        break
    images[name] = result
    if result is not None:
        print("  ✓ Success")

valid = {k: v for k, v in images.items() if v is not None}

if stop:
    pass
elif not valid:
    print("\nNo images generated. The free API limit may have been reached — try again later.")
else:
    fig, axes = plt.subplots(1, len(valid), figsize=(8 * len(valid), 8))
    if len(valid) == 1: axes = [axes]
    for ax, (name, img) in zip(axes, valid.items()):
        ax.imshow(img)
        ax.set_title(name, fontsize=16, pad=20)
        ax.axis("off")
    fig.suptitle(f'Prompt: {PROMPT}', fontsize=18, color="#1a73e8", y=1.05)
    plt.tight_layout()
    plt.show()